## Aplicación simultánea de entropía y coeficiente de Gini

### Preprocesamiento de DataFrame

In [11]:
import sys
from pathlib import Path
import numpy as np
import polars as pl

#Para poder utilizar archivos del proyecto
ROOT = Path.cwd().parent 
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))
from src.cleaning.preprocessing import pipeline_preprocesamiento

df = pipeline_preprocesamiento()

### Entropía de Shannon

In [12]:
def calcular_entropia_shannon(p_array: np.ndarray) -> tuple[float, float, float]:
    k = len(p_array)
    if k <= 1:
        return 0.0, 0.0, 0.0

    p_valid = p_array[p_array > 0]
    shannon_bits = float(-np.sum(p_valid * np.log2(p_valid)))
    shannon_max = float(np.log2(k))
    shannon_rel = float(shannon_bits / shannon_max) if shannon_max > 0 else 0.0

    return shannon_bits, shannon_max, shannon_rel

### Coeficiente de Gini

In [13]:
def calcular_gini_frecuencias(frecuencias_array: np.ndarray) -> float:
    """
    Calcula el Coeficiente de Gini tradicional sobre la distribución de 
    frecuencias/masas poblacionales de las categorías.
    
    Parámetros:
        frecuencias_array: Arreglo NumPy con los conteos o masas ponderadas.
    """
    k = len(frecuencias_array)
    if k <= 1:
        return 0.0

    # Orden ascendente antes de calcular suma ponderada
    x = np.sort(frecuencias_array)
    sum_x = np.sum(x)
    if sum_x == 0:
        return 0.0

    i = np.arange(1, k + 1)
    sum_ix = np.sum(i * x)

    gini = (2 * sum_ix - (k + 1) * sum_x) / (k * sum_x)
    return float(gini)

### Procesamiento Integrador

In [14]:
def analizar_entropia_gini_variable(
    df: pl.DataFrame,
    columna: str = "estrato_socioeconomico",
    peso_col: str | None = "factor_expansion"
) -> pl.DataFrame:
    """
    Agrupa, ordena por frecuencia/masa ponderada y aplica las funciones
    independientes de Entropía de Shannon y Coeficiente de Gini.
    """
    cond = pl.col(columna).is_not_null()
    if peso_col:
        cond = cond & pl.col(peso_col).is_not_null()

    df_filtrado = df.filter(cond)

    # Agregación: Ponderada por peso_col o recuento simple (len)
    if peso_col:
        frecuencias_df = (
            df_filtrado.group_by(columna)
            .agg(n=pl.col(peso_col).sum())
            .sort("n")
            .with_columns(p=pl.col("n") / pl.col("n").sum())
        )
    else:
        frecuencias_df = (
            df_filtrado.group_by(columna)
            .agg(n=pl.len())
            .sort("n")
            .with_columns(p=pl.col("n") / pl.col("n").sum())
        )

    k = frecuencias_df.height
    x_array = frecuencias_df["n"].to_numpy()
    p_array = frecuencias_df["p"].to_numpy()

    # Funciones
    shannon_bits, shannon_max, shannon_rel = calcular_entropia_shannon(p_array)
    gini_frecuencias = calcular_gini_frecuencias(x_array)

    return pl.DataFrame(
        {
            "Variable": [columna],
            "Categorías (K)": [k],
            "Entropía Shannon (bits)": [round(shannon_bits, 4)],
            "Entropía Máxima (bits)": [round(shannon_max, 4)],
            "Entropía Relativa": [f"{shannon_rel * 100:.2f}%"],
            "Gini Frecuencias": [round(gini_frecuencias, 4)],
        }
    )

## Tabla de comparación de medidas

In [15]:
resumen_estrato_pond = analizar_entropia_gini_variable(
    df, 
    columna="estrato_socioeconomico", 
    peso_col="factor_expansion"
)
print("Resultados Ponderados")
print(resumen_estrato_pond)

Resultados Ponderados
shape: (1, 6)
┌────────────────────┬───────────────────┬───────────┬───────────────────┬───────────┬─────────────┐
│ Variable           ┆ Categorías (K)    ┆ Entropía  ┆ Entropía Máxima   ┆ Entropía  ┆ Gini        │
│ ---                ┆ ---               ┆ Shannon   ┆ (bits)            ┆ Relativa  ┆ Frecuencias │
│ str                ┆ i64               ┆ (bits)    ┆ ---               ┆ ---       ┆ ---         │
│                    ┆                   ┆ ---       ┆ f64               ┆ str       ┆ f64         │
│                    ┆                   ┆ f64       ┆                   ┆           ┆             │
╞════════════════════╪═══════════════════╪═══════════╪═══════════════════╪═══════════╪═════════════╡
│ estrato_socioecono ┆ 4                 ┆ 1.7443    ┆ 2.0               ┆ 87.22%    ┆ 0.3144      │
│ mico               ┆                   ┆           ┆                   ┆           ┆             │
└────────────────────┴───────────────────┴───────────┴─